# Quasar: shared-code starter

**Scope:** Okaloosa County, FIPS `12091`. **Default data:** invented exercise sites and weather.

Use this notebook to explore data and explain experiments. The application and this notebook import the same Python implementation. No live cloud request, Mapbox token, notification delivery, or IBM Z connection is needed for the default run.

Before starting, follow `notebooks/README.md` to install the optional notebook environment. Assign one teammate as the owner of this notebook. For a new experiment, copy it under a descriptive name and use your own feature branch.

**Research question:** How do the illustrative site scores change across the synthetic rainfall exercise, and what happens when an input is missing?

Restart the kernel and run all cells from top to bottom before handing this work to another teammate. Keep the source notebook free of saved outputs when committing.


In [ ]:
import sys
from html import escape
from IPython.display import HTML, display

import quasar
from quasar.analysis import scenario_report, compare_scenarios
from quasar.pipeline import collection
from quasar.scoring import score

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Shared package:", quasar.__file__)


If the imports fail, select the Python environment where you ran `python -m pip install -e '.[notebooks]'`. Launching Jupyter from that activated environment is the simplest starting point. No `sys.path` edits or copied versions of the scoring function are needed.

The next helper is presentation code only. Keep reusable data transformations, scoring, validation, and source adapters in `quasar/`, with tests in `tests/`.


In [ ]:
def display_table(records, columns):
    def cell(value):
        if value is None:
            return "Unknown"
        if isinstance(value, list):
            value = ", ".join(str(item) for item in value)
        return escape(str(value))

    header = "".join(f"<th>{escape(name)}</th>" for name in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{cell(row.get(name))}</td>" for name in columns) + "</tr>"
        for row in records
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


## 1. Inspect one scenario

`scenario_report` uses the same `pipeline.collection` and scoring code as the app. The report preserves data mode, time window, model version, source description, and unknown values. Its mean excludes unknown scores.


In [ ]:
report = scenario_report("baseline")
print(report["notice"])
print("Provenance:", report["provenance"])
print("Summary:", report["summary"])
display_table(report["rows"], ["property_id", "label", "rainfall_6h_mm", "score", "band", "missing"])


## 2. Compare exercise windows

The dates are a fixed fictional exercise clock. These are not current forecasts or a historical hurricane replay. The shared comparison helper retains one row per site and scenario; it does not fill missing scores with zero.


In [ ]:
comparison = compare_scenarios()
display_table(comparison, ["property_id", "scenario", "rainfall_6h_mm", "score", "band", "data_status"])

# Verify that this report uses the exact same peak scores as the application's API pipeline.
app_scores = {feature["id"]: feature["properties"]["risk"]["score"] for feature in collection("peak")["features"]}
report_scores = {row["property_id"]: row["score"] for row in comparison if row["scenario"] == "peak"}
assert report_scores == app_scores


## 3. Display a site's scores

This HTML chart is only a view of shared results. Change `selected_site` to another demo ID. Unknown values remain labeled unknown. Bar lengths represent illustrative score points, not flood probability, depth, or a safety classification.


In [ ]:
selected_site = "demo-001"
site_rows = [row for row in comparison if row["property_id"] == selected_site]
assert site_rows, "Choose a property_id from the comparison table."
bars = []
for row in site_rows:
    label = escape(row["scenario"])
    if row["score"] is None:
        bars.append(f"<p>{label}: Unknown (no numeric score)</p>")
    else:
        value = row["score"]
        bars.append(
            f"<p>{label}: {value}/100</p>"
            f'<div style="width:100%;max-width:500px;background:#e6e6e6">'
            f'<div style="width:{value}%;height:16px;background:#238b85"></div></div>'
        )
display(HTML(f"<h3>{escape(selected_site)}: synthetic relative scores</h3>" + "".join(bars)))


## 4. Try an input through shared scoring

Change an input and rerun this cell. The weights, normalization, missing-data behavior and bands live in `quasar/scoring.py`; do not recreate them here. These inputs and coefficients are illustrative and unvalidated.

If an experiment motivates a reusable model change, implement it in the shared module and add tests so both this notebook and the app receive the same change.


In [ ]:
candidate = {
    "rainfall_6h_mm": 70,
    "elevation_m": 2.4,
    "fema_zone": "AE",
    "waterway_distance_m": 200,
    "gauge_margin_m": 0.4,
}
result = score(candidate)
print("Shared baseline:", result["model_version"], result["score"], result["band"])
display_table(result["factors"], ["name", "value", "weight", "contribution"])

missing_terrain = score({**candidate, "elevation_m": None})
assert missing_terrain["score"] is None
print("Missing terrain stays unknown:", missing_terrain)


## 5. Optional: inspect an already imported WeatherNext snapshot

Leave `USE_LOCAL_FORECAST = False` for the reproducible starter run. No cloud query runs in this notebook. To inspect your own snapshot, set `QUASAR_WEATHERNEXT_FILE` to an **absolute path** in the terminal before launching Jupyter, then deliberately enable this cell. This notebook does not load `.env` or print credentials.

The existing CLI importer and access setup are described in `docs/integrations.md`. Stale or missing rainfall must remain unknown. Actual rainfall still combines with fictional terrain, FEMA and gauge inputs in the current mixed demo; it is not a validated property flood model.


In [ ]:
from quasar.adapters.weathernext import load_forecast

USE_LOCAL_FORECAST = False
if USE_LOCAL_FORECAST:
    forecast = load_forecast()
    print("Snapshot status:", forecast["status"])
    if "records" in forecast:
        forecast_report = scenario_report("weathernext")
        print(forecast_report["notice"])
        display_table(forecast_report["rows"], ["property_id", "rainfall_6h_mm", "score", "band", "data_status"])
else:
    print("Local forecast inspection is off. The starter run uses synthetic fixtures only.")


## 6. Handoff checklist

- Write your question, source/run/time window, assumptions, results and limitations in markdown cells.
- Put reusable logic in `quasar/` and regression tests in `tests/`; import that code here.
- Keep credentials and downloaded data out of notebook source and outputs.
- Restart the kernel and run all cells; record dependency additions in the optional notebook setup.
- Run `python scripts/check_notebooks.py --clear` before committing, then `python scripts/check_notebooks.py --execute` in the notebook environment.
- Open a focused pull request into `preview` for teammate review. A notebook finding is not automatically a change to the production scoring policy.

The teammate using regular Python can inspect the same shared report with `python -m quasar.analysis --compare`; Jupyter is optional for that workflow.
